In [1]:
# Define the path to our local dataset
import os

# Define the base directory as current working directory
base_dir = os.getcwd()
final_data_path = os.path.join(base_dir, 'final')

print('Using local data path:', final_data_path)


Using local data path: c:\Users\91892\Desktop\6TH SEM\CL&NLP\final


In [2]:
print(final_data_path)


c:\Users\91892\Desktop\6TH SEM\CL&NLP\final


<a id="basic-setup"></a>
# 📂 Basic Setup

In [3]:
import os
#importing final dataset
file_path = os.path.join(final_data_path, 'en_US', 'en_US.twitter.txt')
if os.path.exists(file_path):
    print('✅ File found:', file_path)
else:
    print('❌ File not found. Check the folder structure.')

# Now you can open and read it
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = f.read()
    print('✅ File reading successful.')
except Exception as e:
    print(f'❌ Error reading file: {e}')

✅ File found: c:\Users\91892\Desktop\6TH SEM\CL&NLP\final\en_US\en_US.twitter.txt
✅ File reading successful.


# In this code cell, we'll import our packages. As we'll implement n-gram models from scratch we'll just use numpy and Additionally nltk  (just for Tokenization).

In [4]:
%%capture

## Importing Packages
import math
import nltk
import random
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## nltk settings
nltk.download('punkt')
nltk.download('punkt_tab')

# We already loaded the data in previous cell

<a id="pre-process"></a>
# 🧽 Pre-Processing pipeline

In [5]:
def preprocess_pipeline(data) -> 'list':

    # Split by newline character
    sentences = data.split('\n')

    # Remove leading and trailing spaces
    sentences = [s.strip() for s in sentences]

    # Drop Empty Sentences
    sentences = [s for s in sentences if len(s) > 0]

    # Empty List to hold Tokenized Sentences
    tokenized = []

    # Iterate through sentences
    for sentence in sentences:

        # Convert to lowercase
        sentence = sentence.lower()

        # Convert to a list of words
        token = nltk.word_tokenize(sentence)

        # Append to list
        tokenized.append(token)

    return tokenized


## Pass our data to this function
tokenized_sentences = preprocess_pipeline(data)

<a id="split"></a>
# ✂️ Splitting into Train, Valid and Test

In [6]:
## Obtain Train and Test Split
train, test = train_test_split(tokenized_sentences, test_size=0.2, random_state=42)

## Obtain Train and Validation Split
train, val = train_test_split(train, test_size=0.25, random_state=42)

<a id="clean"></a>
# 🧹 Cleaning the Data

<a id="frequency"></a>
## 📔 Creating a Frequency Dictionary

As our dataset is quite big, we'll only use those words that appear `k` times in our dataset. In this function, we'll create a frequency dictionary for our vocabulary.

In [7]:
def count_the_words(sentences) -> 'dict':

  # Creating a Dictionary of counts
  word_counts = {}

  # Iterating over sentences
  for sentence in sentences:

    # Iterating over Tokens
    for token in sentence:

      # Add count for new word
      if token not in word_counts.keys():
        word_counts[token] = 1

      # Increase count by one
      else:
        word_counts[token] += 1

  return word_counts

<a id="closed"></a>
## 🔒 Creating a Closed Vocabulary

One of the most essential steps in dealing with Textual data is handling Out-of-vocabulary words. This helps the model to handle words which are not present in the training corpus. First step in this process is to create a `closed_vocabulary`. This function creates a closed vocabulary containing only those words according to the `count_threshold` parameter.

In [8]:
def handling_oov(tokenized_sentences, count_threshold) -> 'list':

  # Empty list for closed vocabulary
  closed_vocabulary = []

  # Obtain frequency dictionary using previously defined function
  words_count = count_the_words(tokenized_sentences)

  # Iterate over words and counts
  for word, count in words_count.items():

    # Append if it's more(or equal) to the threshold
    if count >= count_threshold :
      closed_vocabulary.append(word)

  return closed_vocabulary

<a id="unk"></a>
## 🤷🏻 Adding UNK Tokens

In this function we'll add `<unk>` tokens, to those words which are not in the `closed_vocabulary` which we just made.

In [9]:
def unk_tokenize(tokenized_sentences, vocabulary, unknown_token = "<unk>") -> 'list':
  #unk tokenize function replaces all tokens not in the vocabulary with <unk> token
  # Convert Vocabulary into a set
  vocabulary = set(vocabulary)

  # Create empty list for sentences
  new_tokenized_sentences = []

  # Iterate over sentences
  for sentence in tokenized_sentences:

    # Iterate over sentence and add <unk>
    # if the token is absent from the vocabulary
    new_sentence = []
    for token in sentence:
      if token in vocabulary:
        new_sentence.append(token)
      else:
        new_sentence.append(unknown_token)

    # Append sentece to the new list
    new_tokenized_sentences.append(new_sentence)

  return new_tokenized_sentences

<a id="final"></a>
## 🧼 Final Cleaning Pipeline

In [10]:
def cleansing(train_data, test_data, count_threshold):

  # Get closed Vocabulary
  vocabulary = handling_oov(train_data, count_threshold)

  # Updated Training Dataset
  new_train_data = unk_tokenize(train_data, vocabulary)

  # Updated Test Dataset
  new_test_data = unk_tokenize(test_data, vocabulary)

  return new_train_data, new_test_data, vocabulary

In [12]:
min_freq = 6 #minimum frequency threshold for vocabulary
final_train, final_test, vocabulary = cleansing(train, test, min_freq)

<a id="build"></a>
# 💪🏻 Building The "Model"

This is a helper function, which will come in handy during inference. This function returns a mapping from n-grams to their frequency in the dataset.

In [13]:
def count_n_grams(data, n, start_token = "<s>", end_token = "<e>") -> 'dict':

  # Empty dict for n-grams
  n_grams = {}

  # Iterate over all sentences in the dataset
  for sentence in data:

    # Append n start tokens and a single end token to the sentence
    sentence = [start_token]*n + sentence + [end_token]

    # Convert the sentence into a tuple
    sentence = tuple(sentence)

    # Temp var to store length from start of n-gram to end
    m = len(sentence) if n==1 else len(sentence)-1

    # Iterate over this length
    for i in range(m):

      # Get the n-gram
      n_gram = sentence[i:i+n]

      # Add the count of n-gram as value to our dictionary
      # IF n-gram is already present
      if n_gram in n_grams.keys():
        n_grams[n_gram] += 1
      # Add n-gram count
      else:
        n_grams[n_gram] = 1

  return n_grams

This function calculates the priority for the next word given the prior n-gram. This function also implements k-smoothing which helps account for unseen n-grams. Using the previously defined formula:


$$\large
P(w_n|w_{n−N+1:n−1}) = \frac{C(w_{n−N+1:n−1}w_n)}{C(w_{n−N+1:n−1})}
$$

---

### K-smoothing

But what if we come across a n-gram that wasn't in the training set. Then our denominator would would become zero and our definition of probability will become invalid. Thus, we use k-smoothing, which adds a positive constant $k$ to each numerator and $k \times |V|$ in the denominator, where $|V|$ is the number of words in the vocabulary. This ensures any n-gram with zero count has the same probability of $\frac{1}{|V|}$. Thus, our original estimation get's modified to:

$$\large
P(w_n|w_{n−N+1:n−1}) = \frac{C(w_{n−N+1:n−1}w_n) + k}{C(w_{n−N+1:n−1} + k |V|)}
$$

In [14]:
def prob_for_single_word(word, previous_n_gram, n_gram_counts, nplus1_gram_counts, vocabulary_size, k = 1.0) -> 'float':
  #k smoothing function to handle unseen n-grams
  # Convert the previous_n_gram into a tuple
  previous_n_gram = tuple(previous_n_gram)

  # Calculating the count, if exists from our freq dictionary otherwise zero
  previous_n_gram_count = n_gram_counts[previous_n_gram] if previous_n_gram in n_gram_counts else 0

  # The Denominator
  denom = previous_n_gram_count + k * vocabulary_size

  # previous n-gram plus the current word as a tuple
  nplus1_gram = previous_n_gram + (word,)

  # Calculating the nplus1 count, if exists from our freq dictionary otherwise zero
  nplus1_gram_count = nplus1_gram_counts[nplus1_gram] if nplus1_gram in nplus1_gram_counts else 0

  # Numerator
  num = nplus1_gram_count + k

  # Final Fraction
  prob = num / denom
  return prob

Now, we loop over all the words in the vocabulary and then compute their probabilites using our `prob_for_single_word()` fn.

In [15]:
def probs(previous_n_gram, n_gram_counts, nplus1_gram_counts, vocabulary, k=1.0) -> 'dict':

  # Convert to Tuple
  previous_n_gram = tuple(previous_n_gram)

  # Add end and unknown tokens to the vocabulary
  vocabulary = vocabulary + ["<e>", "<unk>"]

  # Calculate the size of the vocabulary
  vocabulary_size = len(vocabulary)

  # Empty dict for probabilites
  probabilities = {}

  # Iterate over words
  for word in vocabulary:

    # Calculate probability
    probability = prob_for_single_word(word, previous_n_gram,
                                           n_gram_counts, nplus1_gram_counts,
                                           vocabulary_size, k=k)
    # Create mapping: word -> probability
    probabilities[word] = probability

  return probabilities

<a id="auto-complete"></a>
# 💬 The Auto-Complete System

Finally, we build our `auto_complete` fn. We simply loop over all the words in the vocabulary assuming that they can be the next word and then return the word with it's probability.

In [16]:
def auto_complete(previous_tokens, n_gram_counts, nplus1_gram_counts, vocabulary, k=1.0, start_with=None):


    # length of previous words
    n = len(list(n_gram_counts.keys())[0])

    # most recent 'n' words
    previous_n_gram = previous_tokens[-n:]

    # Calculate probabilty for all words
    probabilities = probs(previous_n_gram,n_gram_counts, nplus1_gram_counts,vocabulary, k=k)

    # Intialize the suggestion and max probability
    suggestion = None
    max_prob = 0

    # Iterate over all words and probabilites, returning the max.
    # We also add a check if the start_with parameter is provided
    for word, prob in probabilities.items():

        if start_with != None:

            if not word.startswith(start_with):
                continue

        if prob > max_prob:

            suggestion = word
            max_prob = prob

    return suggestion, max_prob

We can also loop over all the various n-gram models to get multiple suggestions. This function just extends from the previously defined function by **taking multiple n-gram counts** instead of one. This allows us to take unigram, bigram, .. counts into account as well.

In [17]:
def get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with=None):

    # See how many models we have
    count = len(n_gram_counts_list)

    # Empty list for suggestions
    suggestions = []

    # IMP: Earlier "-1"

    # Loop over counts
    for i in range(count-1):

        # get n and nplus1 counts
        n_gram_counts = n_gram_counts_list[i]
        nplus1_gram_counts = n_gram_counts_list[i+1]

        # get suggestions
        suggestion = auto_complete(previous_tokens, n_gram_counts,
                                    nplus1_gram_counts, vocabulary,
                                    k=k, start_with=start_with)
        # Append to list
        suggestions.append(suggestion)

    return suggestions

<a id="inference"></a>
# 😊 Inference

Here, we create a list of n-gram counts for a arbitrary range `(1,6)`

In [18]:
n_gram_counts_list = []
for n in range(1, 6):
    n_model_counts = count_n_grams(final_train, n)
    n_gram_counts_list.append(n_model_counts)

Let's give it a sample input of "i was about" in a tokenized manner and get multiple suggestions using the above calculated n-gram counts with smoothing-factor, `k` = 1.0

In [19]:
previous_tokens = ["i", "was", "about"]
suggestion = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

display(suggestion)

[('the', 0.05352633604626057),
 ('to', 0.0028051829093754172),
 ('to', 0.0019167002089203228),
 ('lol', 1.92130341223486e-05)]

<a id="misc"></a>
# 🧐 Miscellaneous

Let's see how many n-grams we have in our corpus.

In [20]:
print("unigram count:" , len(n_gram_counts_list[0]))
print("bigram count:", len(n_gram_counts_list[1]))
print("trigram count:", len(n_gram_counts_list[2]))
print("quadgram count:", len(n_gram_counts_list[3]))
print("quintgram count:", len(n_gram_counts_list[4]))

unigram count: 52049
bigram count: 2888207
trigram count: 9857011
quadgram count: 15716332
quintgram count: 19119342


In this section, we just export this list to a `.txt` file so that we can use this for inference rather than "training" each time.

In [21]:
# Storing to file
with open("en_counts.txt", 'wb') as f:
    pickle.dump(n_gram_counts_list, f)

In [ ]:
# # Storing to file
# with open("vocab.txt", 'wb') as f:
#     pickle.dump(vocabulary, f)

# 🧪 Testing the N-Gram Model

Let's create a function that allows us to interactively test our N-Gram model. This function will take a sentence as input, tokenize it, and then predict the next word using our trained model.

In [22]:
def test_n_gram_model(input_text, n_gram_counts_list=n_gram_counts_list, vocabulary=vocabulary, k=1.0, num_predictions=5):
    """
    Test the N-gram model by predicting the next word for a given input text.
    
    Args:
        input_text (str): The input text/sentence for which to predict the next word
        n_gram_counts_list (list): List of n-gram count dictionaries
        vocabulary (list): The vocabulary list
        k (float): The smoothing parameter
        num_predictions (int): Number of top predictions to return
    
    Returns:
        list: Top predicted next words with their probabilities
    """
    # Tokenize the input text
    tokens = nltk.word_tokenize(input_text.lower())
    
    # Replace OOV words with <unk>
    tokens = ['<unk>' if token not in vocabulary else token for token in tokens]
    
    # Get the maximum n-gram size from our model counts
    max_n = len(n_gram_counts_list)
    
    # If we have fewer tokens than our maximum n-gram size,
    # we'll use what we have and pad with start tokens if needed
    context_size = min(max_n, len(tokens))
    context = tokens[-context_size:]
    
    # Pad with start tokens if needed
    if context_size < max_n:
        context = ['<s>'] * (max_n - context_size) + context
    
    print(f"\nContext: {' '.join(context)}")
    
    # Get all possible next words with their probabilities
    probabilities = {}
    
    # We'll use the largest n-gram model for prediction
    n = len(n_gram_counts_list)
    n_gram_counts = n_gram_counts_list[n-2]  # n-1 gram counts
    nplus1_gram_counts = n_gram_counts_list[n-1]  # n gram counts
    
    # Get previous n-gram (context)
    previous_n_gram = tuple(context[-(n-1):])
    
    # Get probabilities for all words in vocabulary
    all_probs = probs(previous_n_gram, n_gram_counts, nplus1_gram_counts, vocabulary, k=k)
    
    # Sort words by probability and get top predictions
    sorted_probs = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)
    top_predictions = sorted_probs[:num_predictions]
    
    return top_predictions

In [23]:
def interactive_test():
    """Interactive testing loop for the N-gram model"""
    print("Welcome to the N-Gram Auto-Completion Tester!")
    print("Enter a sentence or phrase and the model will predict the next word.")
    print("Type 'quit' to exit.")
    
    while True:
        user_input = input("\nEnter text: ")
        
        if user_input.lower() == 'quit':
            print("Thank you for testing!")
            break
        
        if not user_input:
            print("Please enter some text!")
            continue
        
        # Get predictions
        predictions = test_n_gram_model(user_input)
        
        # Display predictions
        print("\nPredicted next words:")
        for i, (word, prob) in enumerate(predictions, 1):
            print(f"{i}. {word} (probability: {prob:.6f})")

In [24]:
# Let's test with some example sentences
test_sentences = [
    "i want to go",
    "she said that she",
    "the weather is",
    "i was about"
]

for sentence in test_sentences:
    print(f"\n===== Testing: '{sentence}' =====")
    predictions = test_n_gram_model(sentence)
    print("Predicted next words:")
    for i, (word, prob) in enumerate(predictions, 1):
        print(f"{i}. {word} (probability: {prob:.6f})")


===== Testing: 'i want to go' =====

Context: <s> i want to go
Predicted next words:
1. to (probability: 0.003032)
2. back (probability: 0.000648)
3. home (probability: 0.000572)
4. on (probability: 0.000324)
5. out (probability: 0.000305)

===== Testing: 'she said that she' =====

Context: <s> she said that she
Predicted next words:
1. 's (probability: 0.000058)
2. was (probability: 0.000038)
3. lol (probability: 0.000019)
4. my (probability: 0.000019)
5. fave (probability: 0.000019)

===== Testing: 'the weather is' =====

Context: <s> <s> the weather is
Predicted next words:
1. so (probability: 0.000096)
2. amazing (probability: 0.000077)
3. nice (probability: 0.000077)
4. perfect (probability: 0.000077)
5. just (probability: 0.000058)

===== Testing: 'i was about' =====

Context: <s> <s> i was about
Predicted next words:
1. to (probability: 0.000480)
2. order (probability: 0.000038)
3. too (probability: 0.000038)
4. 140 (probability: 0.000038)
5. lol (probability: 0.000019)


In [ ]:
#interactive_test()

# 🚀 Improving the Model Performance

Let's enhance our model by incorporating more data and optimizing parameters.

In [25]:
# Load additional data sources (blogs and news) to improve the model
def load_multiple_sources(base_path, lang='en_US', sources=['blogs', 'news', 'twitter'], max_lines=None):
    """
    Load data from multiple source files for a given language.
    
    
    Args:
        base_path: Base directory path
        lang: Language code directory
        sources: List of source types to load (blogs, news, twitter)
        max_lines: Maximum number of lines to load from each source (None for all)
    
    Returns:
        Combined text data from all sources
    """
    combined_data = ""
    
    for source in sources:
        file_path = os.path.join(base_path, lang, f"{lang}.{source}.txt")
        print(f"Loading {file_path}...")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                if max_lines:
                    lines = [next(f) for _ in range(max_lines)]
                    source_data = '\n'.join(lines)
                else:
                    source_data = f.read()
                    
                lines = source_data.split('\n')
                print(f" - Loaded {len(lines)} lines from {source}")
                combined_data += source_data + "\n"
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
    
    return combined_data

In [26]:
# Load a balanced subset from all three sources
# Using max_lines to keep memory usage manageable
enhanced_data = load_multiple_sources(final_data_path, sources=['blogs', 'news', 'twitter'], max_lines=100000)

print(f"Total data size: {len(enhanced_data)} characters")

# Process the enhanced dataset
enhanced_tokenized = preprocess_pipeline(enhanced_data)
print(f"Total sentences after preprocessing: {len(enhanced_tokenized)}")

# Split into train/test sets
enhanced_train, enhanced_test = train_test_split(
    enhanced_tokenized, test_size=0.2, random_state=42)

# Clean the data with lower threshold for better vocabulary coverage
min_freq = 3  # Reduced from 6 to include more words
enhanced_train_final, enhanced_test_final, enhanced_vocabulary = cleansing(enhanced_train, enhanced_test, min_freq)

print(f"Enhanced vocabulary size: {len(enhanced_vocabulary)}")

Loading c:\Users\91892\Desktop\6TH SEM\CL&NLP\final\en_US\en_US.blogs.txt...
 - Loaded 200000 lines from blogs
Loading c:\Users\91892\Desktop\6TH SEM\CL&NLP\final\en_US\en_US.news.txt...
 - Loaded 200000 lines from news
Loading c:\Users\91892\Desktop\6TH SEM\CL&NLP\final\en_US\en_US.twitter.txt...
 - Loaded 200000 lines from twitter
Total data size: 50539483 characters
Total sentences after preprocessing: 300000
Enhanced vocabulary size: 57950


In [27]:
# Build n-gram counts for the enhanced model
enhanced_n_gram_counts_list = []
for n in range(1, 6):
    print(f"Building {n}-gram model...")
    n_model_counts = count_n_grams(enhanced_train_final, n)
    enhanced_n_gram_counts_list.append(n_model_counts)
    print(f"  - {n}-gram count: {len(n_model_counts)}")

Building 1-gram model...
  - 1-gram count: 57953
Building 2-gram model...
  - 2-gram count: 1779031
Building 3-gram model...
  - 3-gram count: 4996527
Building 4-gram model...
  - 4-gram count: 7122223
Building 5-gram model...
  - 5-gram count: 8021131


In [ ]:
# Save the enhanced model
with open("enhanced_en_counts.txt", 'wb') as f:
    pickle.dump(enhanced_n_gram_counts_list, f)

with open("enhanced_vocab.txt", 'wb') as f:
    pickle.dump(enhanced_vocabulary, f)

## Optimizing Model Parameters

Let's experiment with different smoothing parameters to find the optimal one for our model.

In [28]:
def evaluate_perplexity(test_sentences, n_gram_counts_list, vocabulary, k=1.0):
    """
    Evaluate model performance using perplexity on test data.
    Lower perplexity indicates better model performance.
    
    Args:
        test_sentences: List of tokenized test sentences
        n_gram_counts_list: List of n-gram count dictionaries
        vocabulary: The vocabulary list
        k: Smoothing parameter
        
    Returns:
        Average perplexity across test sentences
    """
    n = len(n_gram_counts_list)
    n_gram_counts = n_gram_counts_list[n-2]  # (n-1)-gram counts
    nplus1_gram_counts = n_gram_counts_list[n-1]  # n-gram counts
    
    total_log_prob = 0
    total_tokens = 0
    
    for sentence in test_sentences[:100]:  # Limit to 100 sentences for efficiency
        # Add start and end tokens
        sentence = ['<s>'] * (n-1) + sentence + ['<e>']
        
        # Calculate probability of each word given previous n-1 words
        for i in range(n-1, len(sentence)):
            previous_n_gram = tuple(sentence[i-(n-1):i])
            word = sentence[i]
            
            # Get probability of this word
            word_prob = prob_for_single_word(word, previous_n_gram, 
                                             n_gram_counts, nplus1_gram_counts, 
                                             len(vocabulary)+2, k)
            
            # Add log probability
            total_log_prob += math.log2(word_prob)
            total_tokens += 1
    
    # Calculate perplexity
    avg_log_prob = total_log_prob / total_tokens
    perplexity = 2 ** (-avg_log_prob)
    
    return perplexity

# Test different smoothing values
k_values = [0.01, 0.1, 0.5, 1.0, 1.5, 2.0, 5.0]
perplexities = []

for k in k_values:
    perplexity = evaluate_perplexity(enhanced_test_final, enhanced_n_gram_counts_list, enhanced_vocabulary, k)
    perplexities.append(perplexity)
    print(f"Smoothing k={k}, Perplexity={perplexity:.2f}")

# Find best k value
best_k_index = np.argmin(perplexities)
best_k = k_values[best_k_index]
print(f"\nBest smoothing parameter: k={best_k} (Perplexity: {perplexities[best_k_index]:.2f})")

Smoothing k=0.01, Perplexity=31498.08
Smoothing k=0.1, Perplexity=36975.05
Smoothing k=0.5, Perplexity=41553.75
Smoothing k=1.0, Perplexity=43526.38
Smoothing k=1.5, Perplexity=44639.92
Smoothing k=2.0, Perplexity=45406.07
Smoothing k=5.0, Perplexity=47702.35

Best smoothing parameter: k=0.01 (Perplexity: 31498.08)


In [29]:
def test_enhanced_model(input_text, n_gram_counts_list=enhanced_n_gram_counts_list, vocabulary=enhanced_vocabulary, k=None, num_predictions=5):
    """
    Test the enhanced N-gram model by predicting the next word.
    
    Args:
        input_text: The input text/sentence for which to predict the next word
        n_gram_counts_list: List of n-gram count dictionaries
        vocabulary: The vocabulary list
        k: The smoothing parameter (if None, use the best k determined by perplexity evaluation)
        num_predictions: Number of top predictions to return
    """
    if k is None:
        # Use the best k from our experiments, or default to 1.0
        try:
            k = best_k
        except NameError:
            k = 1.0
    
    # Tokenize the input text
    tokens = nltk.word_tokenize(input_text.lower())
    
    # Replace OOV words with <unk>
    tokens = ['<unk>' if token not in vocabulary else token for token in tokens]
    
    # Get the maximum n-gram size from our model counts
    max_n = len(n_gram_counts_list)
    
    # If we have fewer tokens than our maximum n-gram size,
    # we'll use what we have and pad with start tokens if needed
    context_size = min(max_n, len(tokens))
    context = tokens[-context_size:]
    
    # Pad with start tokens if needed
    if context_size < max_n:
        context = ['<s>'] * (max_n - context_size) + context
    
    print(f"\nContext: {' '.join(context)}")
    
    # We'll use the largest n-gram model for prediction
    n = len(n_gram_counts_list)
    n_gram_counts = n_gram_counts_list[n-2]  # n-1 gram counts
    nplus1_gram_counts = n_gram_counts_list[n-1]  # n gram counts
    
    # Get previous n-gram (context)
    previous_n_gram = tuple(context[-(n-1):])
    
    # Get probabilities for all words in vocabulary
    all_probs = probs(previous_n_gram, n_gram_counts, nplus1_gram_counts, vocabulary, k=k)
    
    # Sort words by probability and get top predictions
    sorted_probs = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)
    top_predictions = sorted_probs[:num_predictions]
    
    print("Predicted next words:")
    for i, (word, prob) in enumerate(top_predictions, 1):
        print(f"{i}. {word} (probability: {prob:.6f})")
    
    return top_predictions

In [30]:
# Compare original model with enhanced model on the same test sentences
test_sentences = [
    "i want to go",
    "she said that she",
    "the weather is",
    "i was about"
]

for sentence in test_sentences:
    print(f"\n===== Testing: '{sentence}' =====\n")
    
    print("ORIGINAL MODEL PREDICTIONS:")
    test_n_gram_model(sentence, n_gram_counts_list, vocabulary)
    
    print("\nENHANCED MODEL PREDICTIONS:")
    test_enhanced_model(sentence)
    
    print("\n" + "-"*50)


===== Testing: 'i want to go' =====

ORIGINAL MODEL PREDICTIONS:

Context: <s> i want to go

ENHANCED MODEL PREDICTIONS:

Context: <s> i want to go
Predicted next words:
1. to (probability: 0.018958)
2. back (probability: 0.012644)
3. home (probability: 0.009487)
4. out (probability: 0.006330)
5. . (probability: 0.004751)

--------------------------------------------------

===== Testing: 'she said that she' =====

ORIGINAL MODEL PREDICTIONS:

Context: <s> she said that she

ENHANCED MODEL PREDICTIONS:

Context: <s> she said that she
Predicted next words:
1. is (probability: 0.003427)
2. had (probability: 0.001722)
3. told (probability: 0.001722)
4. likely (probability: 0.001722)
5. found (probability: 0.001722)

--------------------------------------------------

===== Testing: 'the weather is' =====

ORIGINAL MODEL PREDICTIONS:

Context: <s> <s> the weather is

ENHANCED MODEL PREDICTIONS:

Context: <s> <s> the weather is
Predicted next words:
1. crazy (probability: 0.003433)
2. goin

In [ ]:
def interactive_enhanced_test():
    """Interactive testing loop for the enhanced N-gram model"""
    print("Welcome to the Enhanced N-Gram Auto-Completion Tester!")
    print("Enter a sentence or phrase and the model will predict the next word.")
    print("Type 'quit' to exit.")
    
    while True:
        user_input = input("\nEnter text: ")
        
        if user_input.lower() == 'quit':
            print("Thank you for testing!")
            break
        
        if not user_input:
            print("Please enter some text!")
            continue
        
        # Get predictions using enhanced model
        test_enhanced_model(user_input)

# Uncomment to run interactive test with enhanced model
interactive_enhanced_test()

Welcome to the Enhanced N-Gram Auto-Completion Tester!
Enter a sentence or phrase and the model will predict the next word.
Type 'quit' to exit.
Thank you for testing!
